# Modèle NLP - Extraction des Destinations (NER)

Ce notebook entraîne un modèle **Spacy NER** (Named Entity Recognition) pour extraire les destinations de **départ** et d'**arrivée** à partir de phrases de trajet valides.

## Objectif
Créer un système qui extrait automatiquement les entités `DEPARTURE` et `ARRIVAL` des phrases de trajet, après validation par le classifieur.

## Structure
1. **Configuration & Imports** - Paramètres centralisés et dépendances
2. **Chargement des Données** - Lecture du dataset JSONL Spacy
3. **Conversion en Format Spacy** - Préparation des données pour l'entraînement
4. **Entraînement du Modèle** - Entraînement du modèle Spacy NER avec checkpoints
5. **Évaluation** - Métriques et analyse des performances
6. **Sauvegarde du Modèle Final** - Export pour production


## 1. Configuration & Imports

### Configuration centralisée

**Objectif** : Définir tous les paramètres en un seul endroit pour faciliter les expérimentations.

**Ce qu'on fait** :
- Définir les chemins vers les fichiers et dossiers (dataset, modèles, résultats, checkpoints)
- Configurer les paramètres du dataset (limite, seed pour reproductibilité)
- Définir comment diviser les données (train/validation/test)
- Configurer les paramètres d'entraînement Spacy (itérations, dropout, etc.)
- Activer/désactiver les checkpoints

**Pourquoi** : Avoir tous les paramètres au même endroit permet de modifier facilement l'expérience sans chercher dans tout le code.


In [23]:
# ============================================================================
# CONFIGURATION - Modifier ces paramètres selon vos besoins
# ============================================================================

import os
from pathlib import Path
from datetime import datetime
from typing import Optional
import json
import random

# Chemins des fichiers
# Le notebook est dans nlp/notebooks/, on remonte de 2 niveaux pour accéder à la racine du projet
PROJECT_ROOT = Path(os.getcwd()).parent.parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
# Utiliser les dossiers spécifiques au NLP
DATASET_PATH = PROJECT_ROOT / "dataset" / "nlp" / "json" / "nlp_training_data.jsonl"
MODELS_DIR = PROJECT_ROOT / "nlp" / "models"
CHECKPOINTS_DIR = PROJECT_ROOT / "nlp" / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "nlp" / "results"
LOGS_DIR = PROJECT_ROOT / "nlp" / "logs" / "training"

# Créer les dossiers si nécessaire
for dir_path in [MODELS_DIR, CHECKPOINTS_DIR, RESULTS_DIR, LOGS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Paramètres du dataset
DATASET_LIMIT = None  # None pour utiliser tout le dataset, ou int pour limiter
RANDOM_SEED = 42

# Paramètres de split
TEST_SIZE = 0.2
VAL_SIZE = 0.1  # Proportion du train qui devient validation

# Paramètres d'entraînement Spacy
N_ITER = 20  # Nombre d'itérations d'entraînement
DROPOUT = 0.1  # Dropout pour éviter le surapprentissage
BATCH_SIZE = 16  # Taille des batches
LEARNING_RATE = 0.001  # Taux d'apprentissage
SAVE_CHECKPOINTS = True  # Activer les checkpoints
CHECKPOINT_INTERVAL = 5  # Sauvegarder tous les N itérations

# Métadonnées
EXPERIMENT_NAME = f"ner_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
VERSION = "v1"

print(f"✅ Configuration chargée")
print(f"   Dataset: {DATASET_PATH}")
print(f"   Expérience: {EXPERIMENT_NAME}")
print(f"   Itérations: {N_ITER}")


✅ Configuration chargée
   Dataset: /workspace/dataset/nlp/json/nlp_training_data.jsonl
   Expérience: ner_model_20260106_101616
   Itérations: 20


## 2. Installation et Imports Spacy

**Objectif** : Installer et importer Spacy, puis charger un modèle de base français.

**Ce qu'on fait** :
- Vérifier l'installation de Spacy
- Charger un modèle français de base (fr_core_news_sm ou fr_core_news_md)
- Définir les labels d'entités (DEPARTURE, ARRIVAL)

**Note** : Si Spacy n'est pas installé, exécutez : `pip install spacy` puis `python -m spacy download fr_core_news_sm`


In [24]:
# Installation de Spacy si nécessaire
try:
    import spacy
    from spacy.training import Example
    from spacy.util import minibatch, compounding
    print("✅ Spacy importé avec succès")
except ImportError:
    print("⚠️  Spacy non installé. Exécutez : pip install spacy")
    print("   Puis : python -m spacy download fr_core_news_sm")

# Labels d'entités à reconnaître
LABELS = ["DEPARTURE", "ARRIVAL"]

# Charger un modèle français de base
# Si fr_core_news_sm n'est pas disponible, essayez fr_core_news_md
try:
    nlp_base = spacy.load("fr_core_news_sm")
    print("✅ Modèle fr_core_news_sm chargé")
except OSError:
    try:
        nlp_base = spacy.load("fr_core_news_md")
        print("✅ Modèle fr_core_news_md chargé")
    except OSError:
        print("⚠️  Aucun modèle français trouvé. Création d'un modèle vide.")
        nlp_base = spacy.blank("fr")
        print("   Pour télécharger un modèle : python -m spacy download fr_core_news_sm")


✅ Spacy importé avec succès
✅ Modèle fr_core_news_sm chargé


## 3. Chargement des Données

**Objectif** : Charger le dataset JSONL et le convertir en format Spacy.

**Ce qu'on fait** :
- Lire le fichier JSONL ligne par ligne
- Parser les entités au format Spacy
- Afficher des statistiques sur le dataset


In [25]:
def load_spacy_jsonl(file_path: Path, limit: Optional[int] = None):
    """Charge un dataset JSONL au format Spacy."""
    data = []
    with file_path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if limit and i >= limit:
                break
            if line.strip():
                item = json.loads(line)
                data.append(item)
    return data

print(f"📂 Chargement du dataset depuis {DATASET_PATH}...")
raw_data = load_spacy_jsonl(DATASET_PATH, limit=DATASET_LIMIT)
print(f"✅ {len(raw_data)} phrases chargées")

# Statistiques
total_entities = sum(len(item["entities"]) for item in raw_data)
departures = sum(1 for item in raw_data for ent in item["entities"] if ent[2] == "DEPARTURE")
arrivals = sum(1 for item in raw_data for ent in item["entities"] if ent[2] == "ARRIVAL")
phrases_with_dep = sum(1 for item in raw_data if any(ent[2] == "DEPARTURE" for ent in item["entities"]))
phrases_with_arr = sum(1 for item in raw_data if any(ent[2] == "ARRIVAL" for ent in item["entities"]))

print(f"\n📊 Statistiques du dataset:")
print(f"   Total d'entités: {total_entities}")
print(f"   Départs (DEPARTURE): {departures}")
print(f"   Arrivées (ARRIVAL): {arrivals}")
print(f"   Phrases avec départ: {phrases_with_dep} ({phrases_with_dep/len(raw_data)*100:.1f}%)")
print(f"   Phrases avec arrivée: {phrases_with_arr} ({phrases_with_arr/len(raw_data)*100:.1f}%)")

# Afficher quelques exemples
print(f"\n📝 Exemples de données:")
for i, item in enumerate(raw_data[:3]):
    print(f"\n   Exemple {i+1}:")
    print(f"   Texte: {item['text']}")
    print(f"   Entités: {item['entities']}")


📂 Chargement du dataset depuis /workspace/dataset/nlp/json/nlp_training_data.jsonl...
✅ 20000 phrases chargées

📊 Statistiques du dataset:
   Total d'entités: 32574
   Départs (DEPARTURE): 15609
   Arrivées (ARRIVAL): 16965
   Phrases avec départ: 15609 (78.0%)
   Phrases avec arrivée: 16964 (84.8%)

📝 Exemples de données:

   Exemple 1:
   Texte: besoin d'aller de la garre DE Clermont La Pardieu a Hoffen rapidement
   Entités: [[18, 49, 'DEPARTURE'], [52, 58, 'ARRIVAL']]

   Exemple 2:
   Texte: Je vais à Calonne-Ricouart.
   Entités: [[10, 26, 'ARRIVAL']]

   Exemple 3:
   Texte: l'aéroport de haponval → la gare de Corcy → l'aéoport de Romilly-sur-Seine
   Entités: [[0, 22, 'DEPARTURE'], [57, 74, 'ARRIVAL']]


## 4. Conversion en Format Spacy

**Objectif** : Convertir les données JSONL en format Spacy Example pour l'entraînement.

**Ce qu'on fait** :
- Créer des objets Example Spacy à partir des données JSONL
- Diviser en train/validation/test
- Préparer les données pour l'entraînement


In [26]:
def convert_to_spacy_examples(data, nlp_model):
    """Convertit les données JSONL en format Spacy Example."""
    import warnings
    
    examples = []
    skipped = 0
    total_entities = 0
    aligned_entities = 0
    
    # Filtrer les warnings d'alignement pour ne pas polluer la sortie
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", category=UserWarning, module="spacy.training.iob_utils")
        
        for item in data:
            text = item["text"]
            entities = item["entities"]
            
            # Créer un doc Spacy
            doc = nlp_model.make_doc(text)
            
            # Créer les annotations d'entités au format Spacy
            # Format pour Example.from_dict : liste de tuples (start, end, label)
            spacy_entities = [(start, end, label) for start, end, label in entities]
            total_entities += len(spacy_entities)
            
            # Vérifier l'alignement des entités avec les tokens
            # Si certaines entités ne peuvent pas être alignées, elles seront ignorées
            try:
                # Créer l'Example avec le format correct
                annotations = {"entities": spacy_entities}
                example = Example.from_dict(doc, annotations)
                
                # Compter les entités alignées
                aligned_count = len(example.reference.ents)
                aligned_entities += aligned_count
                
                # Garder l'exemple même si certaines entités ne sont pas alignées
                # (au moins une entité doit être alignée, ou aucune entité attendue)
                if aligned_count > 0 or len(spacy_entities) == 0:
                    examples.append(example)
                else:
                    skipped += 1
            except Exception as e:
                # En cas d'erreur, on skip cet exemple
                skipped += 1
                continue
    
    print(f"   📊 Statistiques d'alignement:")
    print(f"      Entités totales: {total_entities}")
    print(f"      Entités alignées: {aligned_entities} ({aligned_entities/total_entities*100:.1f}%)")
    if skipped > 0:
        print(f"      ⚠️  {skipped} exemples ignorés (aucune entité alignée)")
    
    return examples

print("🔄 Conversion des données en format Spacy...")
examples = convert_to_spacy_examples(raw_data, nlp_base)
print(f"✅ {len(examples)} exemples convertis")

# Division train/validation/test
random.seed(RANDOM_SEED)
random.shuffle(examples)

n_total = len(examples)
n_test = int(n_total * TEST_SIZE)
n_val = int((n_total - n_test) * VAL_SIZE)
n_train = n_total - n_test - n_val

train_examples = examples[:n_train]
val_examples = examples[n_train:n_train + n_val]
test_examples = examples[n_train + n_val:]

print(f"\n📊 Division des données:")
print(f"   Train: {n_train} ({n_train/n_total*100:.1f}%)")
print(f"   Validation: {n_val} ({n_val/n_total*100:.1f}%)")
print(f"   Test: {n_test} ({n_test/n_total*100:.1f}%)")


🔄 Conversion des données en format Spacy...
   📊 Statistiques d'alignement:
      Entités totales: 32574
      Entités alignées: 32276 (99.1%)
      ⚠️  132 exemples ignorés (aucune entité alignée)
✅ 19868 exemples convertis

📊 Division des données:
   Train: 14306 (72.0%)
   Validation: 1589 (8.0%)
   Test: 3973 (20.0%)


## 5. Préparation du Modèle

**Objectif** : Créer un nouveau modèle Spacy avec le composant NER et ajouter les labels personnalisés.

**Ce qu'on fait** :
- Créer un nouveau modèle à partir du modèle de base
- Ajouter le composant NER s'il n'existe pas
- Ajouter les labels DEPARTURE et ARRIVAL


In [27]:
# Créer un nouveau modèle pour l'entraînement
# Pour Spacy 3.x, on crée un modèle vide et on ajoute le composant NER
# On peut aussi réutiliser le modèle de base, mais il faut gérer le composant NER existant
if nlp_base.meta.get("lang") == "fr":
    # Si on a un modèle français pré-entraîné, on peut le réutiliser
    # Mais on doit créer un nouveau modèle vide pour éviter les conflits avec le NER existant
    nlp = spacy.blank("fr")
else:
    nlp = spacy.blank("fr")

# Ajouter le composant NER (toujours créer un nouveau composant NER)
if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner", last=True)
else:
    # Si le composant existe déjà, on le récupère
    ner = nlp.get_pipe("ner")
    # On peut aussi le remplacer si nécessaire
    # nlp.remove_pipe("ner")
    # ner = nlp.add_pipe("ner", last=True)

# Ajouter les labels personnalisés
for label in LABELS:
    ner.add_label(label)

print(f"✅ Modèle préparé")
print(f"   Labels NER: {ner.labels}")
print(f"   Composants: {nlp.pipe_names}")


✅ Modèle préparé
   Labels NER: ('ARRIVAL', 'DEPARTURE')
   Composants: ['ner']


## 6. Entraînement du Modèle

**Objectif** : Entraîner le modèle Spacy NER avec les données d'entraînement.

**Ce qu'on fait** :
- Désactiver les autres composants pendant l'entraînement
- Entraîner le modèle avec les données d'entraînement
- Évaluer sur la validation à chaque checkpoint
- Sauvegarder les checkpoints intermédiaires


In [28]:
import time
from datetime import timedelta
import numpy as np

def convert_to_python_types(obj):
    """Convertit récursivement les types numpy en types Python natifs pour JSON."""
    if isinstance(obj, (np.integer, np.floating)):
        return float(obj) if isinstance(obj, np.floating) else int(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {key: convert_to_python_types(value) for key, value in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_python_types(item) for item in obj]
    else:
        return obj

def save_checkpoint(nlp_model, checkpoint_num, experiment_name, metrics=None):
    """Sauvegarde un checkpoint du modèle."""
    checkpoint_dir = CHECKPOINTS_DIR / experiment_name
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    checkpoint_path = checkpoint_dir / f"checkpoint_{checkpoint_num:03d}"
    nlp_model.to_disk(checkpoint_path)
    
    if metrics:
        # Convertir les types numpy en types Python pour JSON
        metrics_serializable = convert_to_python_types(metrics)
        metrics_file = checkpoint_dir / f"metrics_{checkpoint_num:03d}.json"
        with metrics_file.open("w", encoding="utf-8") as f:
            json.dump(metrics_serializable, f, indent=2, ensure_ascii=False)
    
    print(f"   💾 Checkpoint {checkpoint_num} sauvegardé: {checkpoint_path}")

def evaluate_model(nlp_model, examples):
    """Évalue le modèle sur un ensemble d'exemples."""
    scorer = nlp_model.evaluate(examples)
    # Convertir les valeurs numpy en float Python pour éviter les problèmes de sérialisation
    per_type = {}
    if scorer.get("ents_per_type"):
        for label, metrics in scorer.get("ents_per_type", {}).items():
            per_type[label] = {
                "p": float(metrics.get("p", 0.0)),
                "r": float(metrics.get("r", 0.0)),
                "f": float(metrics.get("f", 0.0))
            }
    
    return {
        "precision": float(scorer.get("ents_p", 0.0)),
        "recall": float(scorer.get("ents_r", 0.0)),
        "f1": float(scorer.get("ents_f", 0.0)),
        "per_type": per_type
    }

def format_time(seconds):
    """Formate le temps en format lisible."""
    return str(timedelta(seconds=int(seconds)))

def print_progress_bar(iteration, total, length=30):
    """Affiche une barre de progression."""
    percent = (iteration / total) * 100
    filled = int(length * iteration / total)
    bar = "█" * filled + "░" * (length - filled)
    return f"[{bar}] {percent:.1f}%"

print("🚀 Démarrage de l'entraînement...")
print(f"   Itérations: {N_ITER}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Dropout: {DROPOUT}")
print(f"   Données d'entraînement: {len(train_examples)} exemples")
print(f"   Données de validation: {len(val_examples)} exemples")
print()

# Désactiver les autres composants pendant l'entraînement
other_pipes = [pipe for pipe in nlp.pipe_names if pipe != "ner"]
with nlp.disable_pipes(*other_pipes):
    # Initialiser les poids et créer l'optimizer
    optimizer = nlp.begin_training()
    
    training_history = []
    start_time = time.time()
    best_f1 = 0.0
    best_iteration = 0
    
    for iteration in range(N_ITER):
        iter_start_time = time.time()
        
        # Mélanger les données d'entraînement
        random.shuffle(train_examples)
        
        # Créer des batches
        batches = minibatch(train_examples, size=compounding(4.0, 32.0, 1.001))
        losses = {}
        batch_count = 0
        
        # Entraînement
        for batch in batches:
            nlp.update(batch, drop=DROPOUT, losses=losses, sgd=optimizer)
            batch_count += 1
        
        # Évaluation sur la validation
        val_metrics = evaluate_model(nlp, val_examples)
        
        # Calcul du temps
        iter_time = time.time() - iter_start_time
        elapsed_time = time.time() - start_time
        avg_time_per_iter = elapsed_time / (iteration + 1)
        remaining_time = avg_time_per_iter * (N_ITER - iteration - 1)
        
        # Historique (convertir la loss en float Python pour éviter les problèmes de sérialisation)
        history_entry = {
            "iteration": iteration + 1,
            "loss": float(losses.get("ner", 0.0)),
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "per_type": val_metrics["per_type"]
        }
        training_history.append(history_entry)
        
        # Vérifier si c'est le meilleur modèle
        if val_metrics["f1"] > best_f1:
            best_f1 = float(val_metrics["f1"])
            best_iteration = iteration + 1
        
        # Calcul des tendances (comparaison avec l'itération précédente)
        loss_trend = ""
        f1_trend = ""
        if iteration > 0:
            prev_loss = float(training_history[-2]["loss"])
            prev_f1 = float(training_history[-2]["val_f1"])
            if losses.get("ner", 0.0) < prev_loss:
                loss_trend = "📉"
            else:
                loss_trend = "📈"
            if val_metrics["f1"] > prev_f1:
                f1_trend = "📈"
            else:
                f1_trend = "📉"
        
        # Affichage détaillé
        print("=" * 80)
        print(f"   Itération {iteration + 1}/{N_ITER} {print_progress_bar(iteration + 1, N_ITER)}")
        print("=" * 80)
        print(f"   ⏱️  Temps: {format_time(iter_time)} (itération) | {format_time(elapsed_time)} (total) | ~{format_time(remaining_time)} (restant)")
        print(f"   📦 Batches traités: {batch_count}")
        print()
        print(f"   📊 LOSS: {losses.get('ner', 0.0):.4f} {loss_trend}")
        print()
        print(f"   ✅ Métriques de Validation (globales):")
        print(f"      Precision: {val_metrics['precision']:.4f} | Recall: {val_metrics['recall']:.4f} | F1: {val_metrics['f1']:.4f} {f1_trend}")
        print()
        
        # Métriques par type d'entité
        if val_metrics['per_type']:
            print(f"   🏷️  Métriques par type d'entité:")
            for label in LABELS:
                if label in val_metrics['per_type']:
                    label_metrics = val_metrics['per_type'][label]
                    print(f"      {label}:")
                    print(f"         Precision: {label_metrics.get('p', 0.0):.4f} | Recall: {label_metrics.get('r', 0.0):.4f} | F1: {label_metrics.get('f', 0.0):.4f}")
            print()
        
        # Meilleur modèle
        if iteration + 1 == best_iteration:
            print(f"   ⭐ Meilleur modèle jusqu'à présent! (F1: {best_f1:.4f})")
        else:
            print(f"   🏆 Meilleur modèle: Itération {best_iteration} (F1: {best_f1:.4f})")
        
        # Sauvegarder checkpoint
        if SAVE_CHECKPOINTS and (iteration + 1) % CHECKPOINT_INTERVAL == 0:
            print()
            save_checkpoint(nlp, iteration + 1, EXPERIMENT_NAME, history_entry)
        
        print()

print("=" * 80)
print("✅ Entraînement terminé!")
print("=" * 80)
print(f"   Temps total: {format_time(time.time() - start_time)}")
print(f"   Meilleur modèle: Itération {best_iteration} (F1: {best_f1:.4f})")
print()


🚀 Démarrage de l'entraînement...
   Itérations: 20
   Batch size: 16
   Learning rate: 0.001
   Dropout: 0.1
   Données d'entraînement: 14306 exemples
   Données de validation: 1589 exemples

   Itération 1/20 [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░] 5.0%
   ⏱️  Temps: 0:00:21 (itération) | 0:00:21 (total) | ~0:06:55 (restant)
   📦 Batches traités: 1563

   📊 LOSS: 14372.7305 

   ✅ Métriques de Validation (globales):
      Precision: 0.8329 | Recall: 0.9114 | F1: 0.8704 

   🏷️  Métriques par type d'entité:
      DEPARTURE:
         Precision: 0.8337 | Recall: 0.9079 | F1: 0.8692
      ARRIVAL:
         Precision: 0.8321 | Recall: 0.9147 | F1: 0.8715

   ⭐ Meilleur modèle jusqu'à présent! (F1: 0.8704)

   Itération 2/20 [███░░░░░░░░░░░░░░░░░░░░░░░░░░░] 10.0%
   ⏱️  Temps: 0:00:22 (itération) | 0:00:44 (total) | ~0:06:43 (restant)
   📦 Batches traités: 1563

   📊 LOSS: 9335.2236 📉

   ✅ Métriques de Validation (globales):
      Precision: 0.8460 | Recall: 0.8995 | F1: 0.8720 📈

   🏷️  Métrique

## 7. Évaluation sur le Test Set

**Objectif** : Évaluer les performances finales du modèle sur le test set.

**Ce qu'on fait** :
- Évaluer le modèle sur les données de test
- Afficher les métriques détaillées (precision, recall, F1)
- Analyser les performances par type d'entité (DEPARTURE vs ARRIVAL)


In [29]:
print("📊 Évaluation sur le test set...")
test_metrics = evaluate_model(nlp, test_examples)

print(f"\n✅ Résultats sur le test set:")
print(f"   Precision: {test_metrics['precision']:.4f}")
print(f"   Recall: {test_metrics['recall']:.4f}")
print(f"   F1-Score: {test_metrics['f1']:.4f}")

print(f"\n📈 Métriques par type d'entité:")
for label, metrics in test_metrics['per_type'].items():
    print(f"   {label}:")
    print(f"      Precision: {metrics.get('p', 0.0):.4f}")
    print(f"      Recall: {metrics.get('r', 0.0):.4f}")
    print(f"      F1: {metrics.get('f', 0.0):.4f}")

# Sauvegarder les métriques (convertir en types Python pour JSON)
results_dir = RESULTS_DIR / EXPERIMENT_NAME
results_dir.mkdir(parents=True, exist_ok=True)

metrics_file = results_dir / "metrics.json"
metrics_to_save = {
    "test_metrics": test_metrics,
    "training_history": training_history,
    "config": {
        "n_iter": N_ITER,
        "dropout": DROPOUT,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "n_train": n_train,
        "n_val": n_val,
        "n_test": n_test
    }
}
# Convertir tous les types numpy en types Python
metrics_serializable = convert_to_python_types(metrics_to_save)
with metrics_file.open("w", encoding="utf-8") as f:
    json.dump(metrics_serializable, f, indent=2, ensure_ascii=False)

print(f"\n💾 Métriques sauvegardées: {metrics_file}")


📊 Évaluation sur le test set...

✅ Résultats sur le test set:
   Precision: 0.8446
   Recall: 0.8622
   F1-Score: 0.8533

📈 Métriques par type d'entité:
   DEPARTURE:
      Precision: 0.8442
      Recall: 0.8425
      F1: 0.8433
   ARRIVAL:
      Precision: 0.8450
      Recall: 0.8806
      F1: 0.8624

💾 Métriques sauvegardées: /workspace/nlp/results/ner_model_20260106_101616/metrics.json


## 8. Tests sur des Exemples

**Objectif** : Tester le modèle sur quelques exemples pour vérifier visuellement les résultats.

**Ce qu'on fait** :
- Tester le modèle sur des phrases d'exemple
- Afficher les entités détectées
- Comparer avec les annotations attendues


In [30]:
# Tester sur quelques exemples du test set
print("🧪 Tests sur des exemples du test set:\n")

for i, example in enumerate(test_examples[:5]):
    text = example.reference.text
    doc = nlp(text)
    
    print(f"Exemple {i+1}:")
    print(f"   Texte: {text}")
    print(f"   Entités détectées:")
    for ent in doc.ents:
        print(f"      - {ent.text} ({ent.label_}) [{ent.start_char}-{ent.end_char}]")
    
    # Comparer avec les annotations attendues
    expected_entities = example.reference.ents
    print(f"   Entités attendues:")
    for ent in expected_entities:
        print(f"      - {text[ent.start_char:ent.end_char]} ({ent.label_}) [{ent.start_char}-{ent.end_char}]")
    print()


🧪 Tests sur des exemples du test set:

Exemple 1:
   Texte: Je recherche un itinéraire de la gare de illeherviers à l'aéroport de eilhees - roqueedonde
   Entités détectées:
      - la gare de illeherviers (DEPARTURE) [30-53]
      - l'aéroport de eilhees - roqueedonde (ARRIVAL) [56-91]
   Entités attendues:
      - la gare de illeherviers (DEPARTURE) [30-53]
      - l'aéroport de eilhees - roqueedonde (ARRIVAL) [56-91]

Exemple 2:
   Texte: En tgv de l'aéroport de Laigneville à la gare d Plnoët
   Entités détectées:
      - Laigneville (DEPARTURE) [24-35]
      - Plnoët (ARRIVAL) [48-54]
   Entités attendues:
      - l'aéroport de Laigneville (DEPARTURE) [10-35]

Exemple 3:
   Texte: Trajet direct centre de La Croix de Méan centre d messin
   Entités détectées:
      - centre de La Croix de Méan (DEPARTURE) [14-40]
      - centre d messin (ARRIVAL) [41-56]
   Entités attendues:
      - centre de La Croix de Méan (DEPARTURE) [14-40]
      - centre d messin (ARRIVAL) [41-56]

Exemple 4:

## 9. Sauvegarde du Modèle Final

**Objectif** : Sauvegarder le modèle final entraîné pour utilisation en production.

**Ce qu'on fait** :
- Sauvegarder le modèle dans le dossier models/
- Sauvegarder les métadonnées (métriques, configuration)
- Créer un fichier de métadonnées JSON


In [31]:
# Sauvegarder le modèle final
model_name = f"spacy_ner_{EXPERIMENT_NAME}_{VERSION}"
model_path = MODELS_DIR / model_name
nlp.to_disk(model_path)

print(f"✅ Modèle sauvegardé: {model_path}")

# Sauvegarder les métadonnées (convertir en types Python pour JSON)
metadata = {
    "model_name": model_name,
    "experiment_name": EXPERIMENT_NAME,
    "version": VERSION,
    "created_at": datetime.now().isoformat(),
    "test_metrics": test_metrics,
    "config": {
        "n_iter": N_ITER,
        "dropout": DROPOUT,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "n_train": n_train,
        "n_val": n_val,
        "n_test": n_test,
        "labels": LABELS
    }
}
# Convertir tous les types numpy en types Python
metadata_serializable = convert_to_python_types(metadata)
metadata_file = MODELS_DIR / f"metadata_{EXPERIMENT_NAME}_{VERSION}.json"
with metadata_file.open("w", encoding="utf-8") as f:
    json.dump(metadata_serializable, f, indent=2, ensure_ascii=False)

print(f"✅ Métadonnées sauvegardées: {metadata_file}")
print(f"\n📦 Modèle prêt pour la production!")
print(f"   Pour charger: nlp = spacy.load('{model_path}')")


✅ Modèle sauvegardé: /workspace/nlp/models/spacy_ner_ner_model_20260106_101616_v1
✅ Métadonnées sauvegardées: /workspace/nlp/models/metadata_ner_model_20260106_101616_v1.json

📦 Modèle prêt pour la production!
   Pour charger: nlp = spacy.load('/workspace/nlp/models/spacy_ner_ner_model_20260106_101616_v1')


## 10. Test du Modèle Chargé

**Objectif** : Vérifier que le modèle peut être chargé et utilisé correctement.

**Ce qu'on fait** :
- Charger le modèle sauvegardé
- Tester sur quelques phrases d'exemple
- Vérifier que tout fonctionne correctement


In [32]:
# Charger le modèle sauvegardé
print(f"🔄 Chargement du modèle depuis {model_path}...")
nlp_loaded = spacy.load(model_path)
print("✅ Modèle chargé avec succès")

# Tester sur quelques phrases
test_phrases = [
    "Je vais de Paris à Lyon",
    "Billet Marseille Nice demain",
    "Je passe par Bordeaux mais je vais à Toulouse",
    "Trajet la gare de Lille vers l'aéroport de Lyon"
]

print("\n🧪 Tests sur des phrases d'exemple:\n")
for phrase in test_phrases:
    doc = nlp_loaded(phrase)
    print(f"Phrase: {phrase}")
    if doc.ents:
        print("   Entités détectées:")
        for ent in doc.ents:
            print(f"      - {ent.text} ({ent.label_})")
    else:
        print("   Aucune entité détectée")
    print()


🔄 Chargement du modèle depuis /workspace/nlp/models/spacy_ner_ner_model_20260106_101616_v1...
✅ Modèle chargé avec succès

🧪 Tests sur des phrases d'exemple:

Phrase: Je vais de Paris à Lyon
   Entités détectées:
      - Paris (DEPARTURE)
      - Lyon (ARRIVAL)

Phrase: Billet Marseille Nice demain
   Entités détectées:
      - Marseille (DEPARTURE)
      - Nice (ARRIVAL)

Phrase: Je passe par Bordeaux mais je vais à Toulouse
   Entités détectées:
      - Toulouse (ARRIVAL)

Phrase: Trajet la gare de Lille vers l'aéroport de Lyon
   Entités détectées:
      - la gare de Lille (DEPARTURE)
      - l'aéroport de Lyon (ARRIVAL)

